# Fraud Detection | Analysis

### Import Libraries & Data Assets

In [1]:
# Import necessary libraries
import os 
import pandas as pd 
import numpy as np

In [2]:
os.getcwd()
os.chdir("/Users/spencerwillard/code/local/fraud_detection_project")

In [3]:
# Read in data assests
apps = pd.read_csv("data/raw/applications.csv")
outcomes = pd.read_csv("data/raw/outcomes.csv")

In [4]:
print(f"apps shape:{apps.shape}")
print(f"outcomes shape:{outcomes.shape} ")

apps shape:(10000, 3)
outcomes shape:(10000, 4) 


### Merge Datasets

In [5]:
df = apps.merge(outcomes, on="application_id", how="left")

In [6]:
df.shape

(10000, 6)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   application_id       10000 non-null  int64
 1   application_date     10000 non-null  str  
 2   identity_risk_score  10000 non-null  int64
 3   fraud                10000 non-null  int64
 4   account_status       10000 non-null  str  
 5   charged_off          10000 non-null  str  
dtypes: int64(3), str(3)
memory usage: 468.9 KB


df DataFrame looks quite clean so no need to create a .copy() and clean it up.

### Build core summary table 


In [8]:
bins = range(0, 1001, 50)

labels = [
    f"{i}-{i+49}"
    for i in range(0, 1000, 50)
]

df["score_bin"] = pd.cut(
    df["identity_risk_score"],
    bins=bins,
    labels=labels,
    right=False
)

In [9]:
df.head()

,application_id,application_date,identity_risk_score,fraud,account_status,charged_off,score_bin
0,1,2023-01-01 00:00:00,102,0,open,no,100-149
1,2,2023-01-01 01:00:00,435,0,open,no,400-449
2,3,2023-01-01 02:00:00,860,1,prevented,yes,850-899
3,4,2023-01-01 03:00:00,270,0,open,no,250-299
4,5,2023-01-01 04:00:00,106,0,open,no,100-149


In [10]:
# Build out initial summary table
summary = (
    df.groupby("score_bin")
    .agg({
        "application_id": "count",
        "fraud": "sum"
    })
    .reset_index()
    .rename(columns={
        "score_bin": "Score Bins",
        "application_id": "Marginal Apps (n)",
        "fraud": "Marginal Fraud Apps (n)"
    })
    .sort_values(by="Score Bins", ascending=False)
)
summary

,Score Bins,Marginal Apps (n),Marginal Fraud Apps (n)
19,950-999,496,487
18,900-949,554,538
17,850-899,518,481
16,800-849,510,459
15,750-799,492,423
14,700-749,456,337
13,650-699,541,391
12,600-649,503,278
11,550-599,491,200
10,500-549,500,147


In [11]:
summary["Fraud Rate per Bin (%)"] = np.nan
summary["Cumulative Apps (n)"] = np.nan

In [12]:
# Create columns with missing values just to get structure and headers correct
summary["Fraud Rate per Bin (%)"] = np.nan
summary["Cumulative Apps (n)"] = np.nan
summary["Cumulative Fraud Apps (n)"] = np.nan
summary["Recall (%)"] = np.nan
summary["Marginal Clear Apps (n)"] = np.nan
summary["Cumulative Clear Apps (n)"] = np.nan
summary["False Positive Rate (%)"] = np.nan

summary

,Score Bins,Marginal Apps (n),Marginal Fraud Apps (n),Fraud Rate per Bin (%),Cumulative Apps (n),Cumulative Fraud Apps (n),Recall (%),Marginal Clear Apps (n),Cumulative Clear Apps (n),False Positive Rate (%)
19,950-999,496,487,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,900-949,554,538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,850-899,518,481,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,800-849,510,459,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,750-799,492,423,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,700-749,456,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,650-699,541,391,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,600-649,503,278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,550-599,491,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,500-549,500,147,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Compute metrics for columns

total_fraud = summary["Marginal Fraud Apps (n)"].sum() # To use for Recall Calculation

summary["Fraud Rate per Bin (%)"] = (summary["Marginal Fraud Apps (n)"] / summary["Marginal Apps (n)"] *100).round(2)
summary["Cumulative Apps (n)"] = summary["Marginal Apps (n)"].cumsum()
summary["Cumulative Fraud Apps (n)"] = summary["Marginal Fraud Apps (n)"].cumsum()
summary["Recall (%)"] = ((summary["Cumulative Fraud Apps (n)"] / total_fraud) *100).round(2)
summary["Marginal Clear Apps (n)"] = (summary["Marginal Apps (n)"] - summary["Marginal Fraud Apps (n)"])

total_good = summary["Marginal Clear Apps (n)"].sum() # To use for False Positive Rate calculation

summary["Cumulative Clear Apps (n)"] = summary["Marginal Clear Apps (n)"].cumsum()
summary["False Positive Rate (%)"] = ((summary["Cumulative Clear Apps (n)"] / total_good) *100).round(2)


# Check variable totals
print(f"Total Goods: {total_good}")
print(f"Total Frauds: {total_fraud}\n")

# Check completed Summary Table 
summary



Total Goods: 5936
Total Frauds: 4064



,Score Bins,Marginal Apps (n),Marginal Fraud Apps (n),Fraud Rate per Bin (%),Cumulative Apps (n),Cumulative Fraud Apps (n),Recall (%),Marginal Clear Apps (n),Cumulative Clear Apps (n),False Positive Rate (%)
19,950-999,496,487,98.19,496,487,11.98,9,9,0.15
18,900-949,554,538,97.11,1050,1025,25.22,16,25,0.42
17,850-899,518,481,92.86,1568,1506,37.06,37,62,1.04
16,800-849,510,459,90.00,2078,1965,48.35,51,113,1.90
15,750-799,492,423,85.98,2570,2388,58.76,69,182,3.07
14,700-749,456,337,73.90,3026,2725,67.05,119,301,5.07
13,650-699,541,391,72.27,3567,3116,76.67,150,451,7.60
12,600-649,503,278,55.27,4070,3394,83.51,225,676,11.39
11,550-599,491,200,40.73,4561,3594,88.44,291,967,16.29
10,500-549,500,147,29.40,5061,3741,92.05,353,1320,22.24


## Add some visual improvements to summary table

In [14]:
summary = summary.rename(columns={
    "Score Bins": "Score Bin",
    "Marginal Apps (n)": "Apps",
    "Marginal Fraud Apps (n)": "Fraud Apps",
    "Fraud Rate per Bin (%)": "Fraud Rate %",
    "Cumulative Apps (n)": "Cum Apps",
    "Cumulative Fraud Apps (n)": "Cum Fraud",
    "Recall (%)": "Recall %",
    "Marginal Clear Apps (n)": "Clear Apps",
    "Cumulative Clear Apps (n)": "Cum Clear",
    "False Positive Rate (%)": "FPR %"
})

summary

,Score Bin,Apps,Fraud Apps,Fraud Rate %,Cum Apps,Cum Fraud,Recall %,Clear Apps,Cum Clear,FPR %
19,950-999,496,487,98.19,496,487,11.98,9,9,0.15
18,900-949,554,538,97.11,1050,1025,25.22,16,25,0.42
17,850-899,518,481,92.86,1568,1506,37.06,37,62,1.04
16,800-849,510,459,90.00,2078,1965,48.35,51,113,1.90
15,750-799,492,423,85.98,2570,2388,58.76,69,182,3.07
14,700-749,456,337,73.90,3026,2725,67.05,119,301,5.07
13,650-699,541,391,72.27,3567,3116,76.67,150,451,7.60
12,600-649,503,278,55.27,4070,3394,83.51,225,676,11.39
11,550-599,491,200,40.73,4561,3594,88.44,291,967,16.29
10,500-549,500,147,29.40,5061,3741,92.05,353,1320,22.24


In [20]:
styled_summary = summary.style \
    .hide(axis="index") \
    .set_properties(**{"text-align": "center"}) \
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]}
    ])

styled_summary

Score Bin,Apps,Fraud Apps,Fraud Rate %,Cum Apps,Cum Fraud,Recall %,Clear Apps,Cum Clear,FPR %
950-999,496,487,98.190000,496,487,11.980000,9,9,0.150000
900-949,554,538,97.110000,1050,1025,25.220000,16,25,0.420000
850-899,518,481,92.860000,1568,1506,37.060000,37,62,1.040000
800-849,510,459,90.000000,2078,1965,48.350000,51,113,1.900000
750-799,492,423,85.980000,2570,2388,58.760000,69,182,3.070000
700-749,456,337,73.900000,3026,2725,67.050000,119,301,5.070000
650-699,541,391,72.270000,3567,3116,76.670000,150,451,7.600000
600-649,503,278,55.270000,4070,3394,83.510000,225,676,11.390000
550-599,491,200,40.730000,4561,3594,88.440000,291,967,16.290000
500-549,500,147,29.400000,5061,3741,92.050000,353,1320,22.240000


In [16]:
summary.style \
    .hide(axis="index") \
    .format({
        "Fraud Rate %": "{:.2f}",
        "Recall %": "{:.2f}",
        "FPR %": "{:.2f}"
    }) \
    .set_properties(**{"text-align": "center"}) \
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]}
    ])

summary

,Score Bin,Apps,Fraud Apps,Fraud Rate %,Cum Apps,Cum Fraud,Recall %,Clear Apps,Cum Clear,FPR %
19,950-999,496,487,98.19,496,487,11.98,9,9,0.15
18,900-949,554,538,97.11,1050,1025,25.22,16,25,0.42
17,850-899,518,481,92.86,1568,1506,37.06,37,62,1.04
16,800-849,510,459,90.00,2078,1965,48.35,51,113,1.90
15,750-799,492,423,85.98,2570,2388,58.76,69,182,3.07
14,700-749,456,337,73.90,3026,2725,67.05,119,301,5.07
13,650-699,541,391,72.27,3567,3116,76.67,150,451,7.60
12,600-649,503,278,55.27,4070,3394,83.51,225,676,11.39
11,550-599,491,200,40.73,4561,3594,88.44,291,967,16.29
10,500-549,500,147,29.40,5061,3741,92.05,353,1320,22.24
